In [1]:
import numpy as np
import pandas as pd

In [2]:
FOLD = 2

In [3]:
flat_regions_path = f"/scratch1/smaruj/genomic_flat_regions/flat_regions_chrom_states_tsv/fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv"

In [4]:
df = pd.read_csv(flat_regions_path, sep="\t")

In [5]:
import torch

In [6]:
from typing import Optional

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
def splice_and_to_string(main_path: str,
                         slice_path: str,
                         replace_bin_start: int = 295,
                         slice_size = 50,
                         bin_size: int = 2048,
                         trim_bp: int = 131072,
                         alphabet: str = "ACGT",
                         device: Optional[str] = None):
    """
    Load a one-hot DNA tensor (shape: [4, L]), replace one 2048-bp bin in the middle,
    trim `trim_bp` bases from both ends, and convert to an A/C/G/T string.

    Parameters
    ----------
    main_path : str
        Path to the main .pt tensor (e.g. 'chr11_65677312_66988032_X.pt').
    slice_path : str
        Path to the .pt tensor providing the replacement slice
        (e.g. 'chr11_65677312_66988032_slice.pt').
    replace_bin : int
        Index of the 2048-bp bin to replace (default: 320).
    bin_size : int
        Size of each bin in bp (default: 2048).
    trim_bp : int
        Number of bp to cut from each side after replacement (default: 131072).
    alphabet : str
        Order of nucleotides corresponding to the one-hot channels (default: "ACGTN").
        Change if your tensor channel order differs.

    Returns
    -------
    seq_str : str
        The trimmed sequence converted to letters.
    """

    main = torch.load(main_path, map_location=device)
    slc  = torch.load(slice_path, map_location=device)

    if main.dim() != 3 or main.size(1) != len(alphabet):
        raise ValueError(f"Expected main tensor of shape [4, L]. Got {tuple(main.shape)}")

    start = replace_bin_start * bin_size
    end   = (replace_bin_start + slice_size) * bin_size

    if end > main.size(2):
        raise ValueError(f"Replacement range [{start}:{end}] exceeds sequence length {main.size(2)}")
    
    # Check slice sizes
    if slc.dim() != 3 or slc.size(1) != len(alphabet) or slc.size(2) != (end - start):
        raise ValueError(
            f"Slice tensor must have shape [1, {len(alphabet)}, {end-start}], got {tuple(slc.shape)}"
        )

    # Replace
    main[:, :, start:end] = slc

    # Trim
    L = main.size(2)
    if 2 * trim_bp >= L:
        raise ValueError(f"trim_bp ({trim_bp}) is too large for sequence length {L}")
    trimmed = main[:, :, trim_bp : L - trim_bp]

    # Convert to string
    # Argmax over channels -> indices 0..3
    idx = trimmed.argmax(dim=1).cpu().numpy()
    
    letters = np.array(list(alphabet))
    seq_arr = letters[idx]

    if seq_arr.ndim > 1:
        seq_arr = seq_arr.ravel()
    
    seq_str = "".join(seq_arr.tolist())

    return seq_str


In [ ]:
def save_to_fasta(seq: str, fasta_path: str, header: str = "sequence"):
    """
    Save a sequence string to a FASTA file.
    
    Parameters
    ----------
    seq : str
        The sequence (A/C/G/T/N).
    fasta_path : str
        Path to save the FASTA file.
    header : str
        The FASTA header (default: "sequence").
    """
    with open(fasta_path, "w") as f:
        f.write(f">{header}\n")
        # Write 80 characters per line (FASTA convention)
        for i in range(0, len(seq), 80):
            f.write(seq[i:i+80] + "\n")

In [ ]:
for i, row in enumerate(df.itertuples(index=False)):
    
    print(f"seq {i}")
    
    chrom, start, end = row.chrom, row.centered_start, row.centered_end
    
    seq = splice_and_to_string(main_path = f"/scratch1/smaruj/generate_genomic_boundary/ohe_X/fold{FOLD}/{chrom}_{start}_{end}_X.pt",
                         slice_path = f"/scratch1/smaruj/generate_genomic_fountain/results/target_0.5/fold{FOLD}/{chrom}_{start}_{end}_slice.pt",
                         device = device)
    
    save_to_fasta(seq = seq, fasta_path=f"/scratch1/smaruj/alpha_genome_validation/fountain_generation/fold{FOLD}_modified/{chrom}_{start}_{end}.fasta", header=f"{chrom}_{start}_{end}")